# Acc-2: FT + early-stop(hold-out Val) + CRC-VAL test (09 재실험)

**모니터:** NCT hold-out Val 10%만 early-stop  
**보고:** best-val ckpt → CRC-VAL Acc / macro-F1 / corruption  
**금지:** CRC-VAL을 early-stop·모델 선택에 사용

필수 격자: tag × seed{42,43,44} × label_frac{0.10,0.25,1.0} (curriculum/scratch 우선).


## 0. 경로·임포트


In [ ]:
from __future__ import annotations

import json
import random
import shutil
import time
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import amp
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, models, transforms
from torchvision.transforms import functional as TF

ON_KAGGLE = Path("/kaggle/working").exists()
if ON_KAGGLE:
    PROJ_ROOT = Path("/kaggle/working/09_보완_재실험")
else:
    CWD = Path.cwd().resolve()
    PROJ_ROOT = None
    for c in [CWD, CWD.parent, CWD.parent.parent, CWD.parent.parent.parent]:
        if (c / "A_프로토콜").exists() or (c / "notebooks").exists() and (c / "outputs").exists():
            if c.name == "09_보완_재실험" or (c / "A_프로토콜").exists():
                PROJ_ROOT = c
                break
    if PROJ_ROOT is None:
        PROJ_ROOT = CWD

OUT = PROJ_ROOT / "outputs"
DIR_FIG = OUT / "figures"
DIR_MET = OUT / "metrics"
DIR_CKPT = OUT / "checkpoints"
DIR_SAMP = OUT / "samples"
DIR_LOG = OUT / "logs"
DIR_SPLIT = OUT / "splits"
for d in [DIR_FIG, DIR_MET, DIR_CKPT, DIR_SAMP, DIR_LOG, DIR_SPLIT]:
    d.mkdir(parents=True, exist_ok=True)

print("ON_KAGGLE:", ON_KAGGLE)
print("PROJ_ROOT:", PROJ_ROOT)
print("OUT:", OUT)


## 1. 설정


In [ ]:
NCT_CLASS_DIRS = ("ADI", "BACK", "DEB", "LYM", "MUC", "MUS", "NORM", "STR", "TUM")

KAGGLE_NCT_CANDIDATES = [
    Path("/kaggle/input/nct-crc-he-100k-nonorm/NCT-CRC-HE-100K-NONORM"),
    Path("/kaggle/input/nct-crc-he-100k-nonorm"),
    Path("/kaggle/input/datasets/imrankhan77/nct-crc-he-100k-nonorm/NCT-CRC-HE-100K-NONORM"),
    Path("/kaggle/input/nct-crc-he-100k/NCT-CRC-HE-100K"),
    Path("/kaggle/input/nct-crc-he-100k"),
]
KAGGLE_VAL_CANDIDATES = [
    Path("/kaggle/input/crc-val-he-7k/CRC-VAL-HE-7K"),
    Path("/kaggle/input/crc-val-he-7k"),
    Path("/kaggle/input/datasets/imrankhan77/crc-val-he-7k/CRC-VAL-HE-7K"),
]


def _resolve_nct_folder(p: Path) -> Optional[Path]:
    if not p.is_dir():
        return None
    hits = sum(1 for c in NCT_CLASS_DIRS if (p / c).is_dir())
    if hits >= 3:
        return p
    for child in sorted(p.iterdir()):
        if child.is_dir():
            hits = sum(1 for c in NCT_CLASS_DIRS if (child / c).is_dir())
            if hits >= 3:
                return child
    return None


def find_root(cands) -> Optional[Path]:
    for cand in cands:
        r = _resolve_nct_folder(cand)
        if r is not None:
            return r
    root = Path("/kaggle/input")
    if not root.exists():
        return None
    for p in sorted(root.rglob("*")):
        if not p.is_dir():
            continue
        r = _resolve_nct_folder(p)
        if r is None:
            continue
        low = str(p).lower()
        if cands is KAGGLE_VAL_CANDIDATES and ("val" in low or "7k" in low):
            return r
        if cands is KAGGLE_NCT_CANDIDATES and "val" not in low:
            return r
    return None


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


class ImageFolderRGB(Dataset):
    def __init__(self, root: str, size: int, class_subset: Optional[List[str]] = None):
        self.base = datasets.ImageFolder(root)
        if class_subset:
            keep = {i for i, n in enumerate(self.base.classes) if n in class_subset}
            self.indices = [i for i, (_, y) in enumerate(self.base.samples) if y in keep]
            self.remap = {old: new for new, old in enumerate(sorted(keep))}
        else:
            self.indices = list(range(len(self.base)))
            self.remap = {i: i for i in range(len(self.base.classes))}
        self.size = size
        self.to_tensor = transforms.Compose([transforms.Resize((size, size)), transforms.ToTensor()])

    @property
    def num_classes(self) -> int:
        return len(self.remap)

    def __len__(self) -> int:
        return len(self.indices)

    def __getitem__(self, i: int):
        path, y = self.base.samples[self.indices[i]]
        from PIL import Image
        img = Image.open(path).convert("RGB")
        return self.to_tensor(img), self.remap[y]


def stratified_holdout_split(labels: List[int], val_frac: float, seed: int) -> Tuple[List[int], List[int]]:
    rng = np.random.RandomState(seed)
    by_cls: Dict[int, List[int]] = {}
    for i, y in enumerate(labels):
        by_cls.setdefault(int(y), []).append(i)
    train_idx, val_idx = [], []
    for y, idxs in sorted(by_cls.items()):
        idxs = list(idxs)
        rng.shuffle(idxs)
        n_val = max(1, int(round(len(idxs) * val_frac)))
        if n_val >= len(idxs):
            n_val = max(1, len(idxs) // 10)
        val_idx.extend(idxs[:n_val])
        train_idx.extend(idxs[n_val:])
    rng.shuffle(train_idx)
    rng.shuffle(val_idx)
    return train_idx, val_idx


def adopt_shared_split_json(path: Path) -> None:
    if path.exists():
        return
    input_root = Path("/kaggle/input")
    if not input_root.exists():
        return
    hits = list(input_root.rglob(path.name))
    if not hits:
        print("no shared split in /kaggle/input; will create", path.name)
        return
    src = sorted(hits, key=lambda p: p.stat().st_mtime, reverse=True)[0]
    path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(src, path)
    print("copied shared split", src, "->", path)


def load_or_create_split(full: ImageFolderRGB, max_images: int, val_frac: float, split_seed: int) -> dict:
    path = DIR_SPLIT / f"nct_split_seed{split_seed}.json"
    adopt_shared_split_json(path)
    labels = [full.remap[full.base.samples[full.indices[i]][1]] for i in range(len(full))]
    pool = list(range(len(full)))
    rng = random.Random(split_seed)
    rng.shuffle(pool)
    pool = pool[: min(max_images, len(pool))]
    pool_labels = [labels[i] for i in pool]
    if path.exists():
        split = json.loads(path.read_text(encoding="utf-8"))
        print("loaded split", path)
        return split
    local_train, local_val = stratified_holdout_split(pool_labels, val_frac, split_seed)
    train_idx = [pool[i] for i in local_train]
    val_idx = [pool[i] for i in local_val]
    split = {
        "split_seed": split_seed,
        "val_frac": val_frac,
        "max_images": max_images,
        "n_pool": len(pool),
        "n_train": len(train_idx),
        "n_val": len(val_idx),
        "train_idx": train_idx,
        "val_idx": val_idx,
        "note": "CRC-VAL must never appear in this file",
    }
    path.write_text(json.dumps(split, ensure_ascii=False, indent=2), encoding="utf-8")
    print("wrote split", path, "train", len(train_idx), "val", len(val_idx))
    return split


def subsample_by_frac(indices: List[int], labels: List[int], frac: float, seed: int) -> List[int]:
    if frac >= 1.0:
        return list(indices)
    rng = np.random.RandomState(seed)
    by_cls: Dict[int, List[int]] = {}
    for i, y in zip(indices, labels):
        by_cls.setdefault(int(y), []).append(i)
    out = []
    for y, idxs in sorted(by_cls.items()):
        idxs = list(idxs)
        rng.shuffle(idxs)
        n = max(1, int(round(len(idxs) * frac)))
        out.extend(idxs[:n])
    rng.shuffle(out)
    return out


_DETECTED_NCT = find_root(KAGGLE_NCT_CANDIDATES)
_DETECTED_CRC = find_root(KAGGLE_VAL_CANDIDATES)
print("NCT:", _DETECTED_NCT)
print("CRC-VAL(test):", _DETECTED_CRC)


def find_ssl_ckpt(tag: str, seed: int) -> Optional[Path]:
    names = [
        f"ssl_{tag}_seed{seed}_final.pt",
        f"ssl_{tag}_seed{seed}_last.pt",
        f"ssl_{tag}_final.pt",
        f"ssl_{tag}_last.pt",
    ]
    for sr in [DIR_CKPT, Path("/kaggle/input"), Path("/kaggle/working")]:
        if not sr.exists():
            continue
        for name in names:
            hits = list(sr.rglob(name))
            if hits:
                return sorted(hits, key=lambda p: p.stat().st_mtime, reverse=True)[0]
    return None


@dataclass
class FTConfig:
    seed: int = 42
    split_seed: int = 42
    val_frac: float = 0.10
    label_frac: float = 1.0
    run_tag: str = "curriculum"  # curriculum|none|fixed_emphasize|schedule_colorless|macenko_none|scratch|imagenet_init
    data_root: str = field(default_factory=lambda: str(_DETECTED_NCT) if _DETECTED_NCT else "")
    crc_val_root: str = field(default_factory=lambda: str(_DETECTED_CRC) if _DETECTED_CRC else "")
    image_size: int = 224
    max_train_images: int = 50000
    max_crc_images: int = 7180
    num_classes_cap: int = 9
    batch_size: int = 64
    num_workers: int = 2
    ft_epochs: int = 30
    early_stop_patience: int = 5
    lr_ft: float = 1e-3
    weight_decay: float = 1e-4
    backbone: str = "resnet18"
    ssl_ckpt: str = ""
    use_macenko: bool = False
    device: str = field(default_factory=lambda: "cuda" if torch.cuda.is_available() else "cpu")


CFG = FTConfig()
# --- Run switches ---
# CFG.run_tag = "curriculum"
# CFG.run_tag = "scratch"
# CFG.run_tag = "none"
# CFG.run_tag = "fixed_emphasize"
# CFG.run_tag = "schedule_colorless"
# CFG.run_tag = "macenko_none"
# CFG.run_tag = "imagenet_init"
# CFG.seed = 42
# CFG.label_frac = 1.0  # or 0.10, 0.25
CFG.use_macenko = CFG.run_tag == "macenko_none"

if not CFG.ssl_ckpt and CFG.run_tag not in ("scratch", "imagenet_init"):
    found = find_ssl_ckpt(CFG.run_tag, CFG.seed)
    if found:
        CFG.ssl_ckpt = str(found)
        print("auto ssl_ckpt", found)

set_seed(CFG.seed)
DEVICE = torch.device(CFG.device)
assert CFG.data_root and Path(CFG.data_root).exists(), "NCT required"
assert CFG.crc_val_root and Path(CFG.crc_val_root).exists(), "CRC-VAL required as TEST only"
print(json.dumps(asdict(CFG), indent=2, default=str))


## 2. 유틸·로더 (hold-out val vs CRC-VAL test)


In [ ]:
def color_shift(x: torch.Tensor, hue_shift: float = 0.08, sat_scale: float = 0.7) -> torch.Tensor:
    # reuse simple channel ops
    img = x
    img = TF.adjust_hue(img, hue_shift)
    img = TF.adjust_saturation(img, sat_scale)
    return img.clamp(0, 1)


def apply_corrupt(x: torch.Tensor, name: Optional[str]) -> torch.Tensor:
    if name is None or name == "clean":
        return x
    if name == "color_shift":
        return color_shift(x)
    if name == "blur":
        return TF.gaussian_blur(x.unsqueeze(0), kernel_size=9, sigma=2.0).squeeze(0)
    if name == "noise":
        return (x + 0.08 * torch.randn_like(x)).clamp(0, 1)
    if name == "gray":
        g = x.mean(dim=-3, keepdim=True)
        return g.expand_as(x)
    raise ValueError(name)


def try_macenko(x: torch.Tensor) -> torch.Tensor:
    mean = x.mean(dim=(-2, -1), keepdim=True)
    std = x.std(dim=(-2, -1), keepdim=True).clamp_min(1e-4)
    return ((x - mean) / std * 0.2 + 0.5).clamp(0, 1)


class Classifier(nn.Module):
    def __init__(self, num_classes: int, tag: str):
        super().__init__()
        if tag == "imagenet_init":
            try:
                weights = models.ResNet18_Weights.IMAGENET1K_V1
                net = models.resnet18(weights=weights)
            except Exception:
                net = models.resnet18(pretrained=True)
        else:
            try:
                net = models.resnet18(weights=None)
            except TypeError:
                net = models.resnet18(pretrained=False)
        in_dim = net.fc.in_features
        net.fc = nn.Linear(in_dim, num_classes)
        self.net = net

    def forward(self, x):
        return self.net(x)


def load_encoder_into(model: Classifier, ckpt_path: str) -> None:
    ckpt = torch.load(ckpt_path, map_location="cpu")
    state = ckpt.get("encoder", ckpt)
    # Acc-1 Encoder: net.* (fc=Identity) → Classifier.net: strip "net.", keep random fc
    mapped = {}
    for k, v in state.items():
        key = k[4:] if k.startswith("net.") else k
        if key.startswith("fc."):
            continue
        mapped[key] = v
    missing, unexpected = model.net.load_state_dict(mapped, strict=False)
    missing_backbone = [k for k in missing if not k.startswith("fc.")]
    print(
        "loaded ssl encoder; missing", len(missing),
        "unexpected", len(unexpected),
        "missing_backbone", len(missing_backbone),
    )
    if missing_backbone or unexpected:
        raise RuntimeError(
            f"SSL encoder load mismatch: missing_backbone={missing_backbone[:8]} "
            f"unexpected={list(unexpected)[:8]}"
        )


@torch.no_grad()
def evaluate(model, loader, corrupt: Optional[str] = None) -> Dict[str, float]:
    model.eval()
    correct = total = 0
    preds_all, ys_all = [], []
    for x, y in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)
        if CFG.use_macenko:
            x = torch.stack([try_macenko(x[i].cpu()) for i in range(x.size(0))]).to(DEVICE)
        if corrupt:
            x = torch.stack([apply_corrupt(x[i].cpu(), corrupt) for i in range(x.size(0))]).to(DEVICE)
        logits = model(x)
        pred = logits.argmax(1)
        correct += int((pred == y).sum().item())
        total += int(y.numel())
        preds_all.append(pred.cpu())
        ys_all.append(y.cpu())
    y_true = torch.cat(ys_all).numpy()
    y_pred = torch.cat(preds_all).numpy()
    classes = sorted(set(y_true.tolist()))
    f1s = []
    for c in classes:
        tp = np.sum((y_pred == c) & (y_true == c))
        fp = np.sum((y_pred == c) & (y_true != c))
        fn = np.sum((y_pred != c) & (y_true == c))
        prec = tp / (tp + fp + 1e-8)
        rec = tp / (tp + fn + 1e-8)
        f1s.append(2 * prec * rec / (prec + rec + 1e-8))
    return {
        "acc": correct / max(1, total),
        "macro_f1": float(np.mean(f1s)) if f1s else float("nan"),
        "n": total,
    }


probe = datasets.ImageFolder(CFG.data_root)
preferred = [c for c in NCT_CLASS_DIRS if c in probe.classes][: CFG.num_classes_cap]
full = ImageFolderRGB(CFG.data_root, CFG.image_size, class_subset=preferred)
split = load_or_create_split(full, CFG.max_train_images, CFG.val_frac, CFG.split_seed)
train_labels = [full.remap[full.base.samples[full.indices[i]][1]] for i in split["train_idx"]]
ft_train_idx = subsample_by_frac(split["train_idx"], train_labels, CFG.label_frac, CFG.seed)
holdout_val_idx = split["val_idx"]

train_ds = Subset(full, ft_train_idx)
holdout_ds = Subset(full, holdout_val_idx)
crc_full = ImageFolderRGB(CFG.crc_val_root, CFG.image_size, class_subset=preferred)
crc_idx = list(range(min(CFG.max_crc_images, len(crc_full))))
crc_ds = Subset(crc_full, crc_idx)

train_loader = DataLoader(train_ds, batch_size=CFG.batch_size, shuffle=True, num_workers=CFG.num_workers, drop_last=True)
holdout_loader = DataLoader(holdout_ds, batch_size=CFG.batch_size, shuffle=False, num_workers=CFG.num_workers)
crc_loader = DataLoader(crc_ds, batch_size=CFG.batch_size, shuffle=False, num_workers=CFG.num_workers)
NUM_CLASSES = full.num_classes
print(
    "ft_train", len(train_ds),
    "holdout_val", len(holdout_ds),
    "crc_test", len(crc_ds),
    "label_frac", CFG.label_frac,
    "denom=FT_train_full", split["n_train"],
)


## 3. FT + early-stop(hold-out) + CRC-VAL 최종 평가


In [ ]:
tag = CFG.run_tag
model = Classifier(NUM_CLASSES, tag).to(DEVICE)
if tag not in ("scratch", "imagenet_init"):
    assert CFG.ssl_ckpt and Path(CFG.ssl_ckpt).exists(), f"ssl ckpt required for {tag}"
    load_encoder_into(model, CFG.ssl_ckpt)

opt = torch.optim.AdamW(model.parameters(), lr=CFG.lr_ft, weight_decay=CFG.weight_decay)
best_val = -1.0
best_ep = -1
patience_left = CFG.early_stop_patience
history = []
best_path = DIR_CKPT / f"ft_{tag}_seed{CFG.seed}_lf{CFG.label_frac:.2f}_best.pt"
last_path = DIR_CKPT / f"ft_{tag}_seed{CFG.seed}_lf{CFG.label_frac:.2f}_last.pt"

for ep in range(CFG.ft_epochs):
    model.train()
    losses = []
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        if CFG.use_macenko:
            x = torch.stack([try_macenko(x[i].cpu()) for i in range(x.size(0))]).to(DEVICE)
        opt.zero_grad(set_to_none=True)
        loss = F.cross_entropy(model(x), y)
        loss.backward()
        opt.step()
        losses.append(float(loss.item()))
    hold = evaluate(model, holdout_loader, None)
    row = {"epoch": ep, "train_loss": float(np.mean(losses)), "holdout_acc": hold["acc"], "holdout_macro_f1": hold["macro_f1"]}
    history.append(row)
    print(row)
    torch.save({"model": model.state_dict(), "epoch": ep, "config": asdict(CFG), "holdout": hold}, last_path)
    if hold["acc"] > best_val + 1e-12:
        best_val = hold["acc"]
        best_ep = ep
        patience_left = CFG.early_stop_patience
        torch.save({"model": model.state_dict(), "epoch": ep, "config": asdict(CFG), "holdout": hold}, best_path)
        print("  new best ->", best_path)
    else:
        patience_left -= 1
        if patience_left <= 0:
            print("early stop at ep", ep, "best_ep", best_ep)
            break

# Load best and evaluate CRC-VAL (test only)
ckpt = torch.load(best_path, map_location=DEVICE)
model.load_state_dict(ckpt["model"])
crc_clean = evaluate(model, crc_loader, None)
corrupt_scores = {}
for cname in ["color_shift", "blur", "noise", "gray"]:
    corrupt_scores[cname] = evaluate(model, crc_loader, cname)

# last-epoch secondary
last_ckpt = torch.load(last_path, map_location=DEVICE)
model.load_state_dict(last_ckpt["model"])
crc_last = evaluate(model, crc_loader, None)

result = {
    "protocol": "09_v1_holdout_val_crc_test",
    "tag": tag,
    "seed": CFG.seed,
    "label_frac": CFG.label_frac,
    "n_ft_train": len(train_ds),
    "n_ft_train_full": split["n_train"],
    "n_holdout_val": len(holdout_ds),
    "n_crc_test": len(crc_ds),
    "monitor": "holdout_val",
    "test": "crc_val",
    "best_epoch": best_ep,
    "best_holdout_acc": best_val,
    "crc_acc_best": crc_clean["acc"],
    "crc_macro_f1_best": crc_clean["macro_f1"],
    "crc_acc_last": crc_last["acc"],
    "crc_macro_f1_last": crc_last["macro_f1"],
    "corruption_best": {k: v["acc"] for k, v in corrupt_scores.items()},
    "ssl_ckpt": CFG.ssl_ckpt,
    "best_ckpt": str(best_path),
    "history": history,
}
out_json = DIR_MET / f"ft_{tag}_seed{CFG.seed}_lf{CFG.label_frac:.2f}.json"
out_json.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
pd.DataFrame(history).to_csv(DIR_MET / f"ft_hist_{tag}_seed{CFG.seed}_lf{CFG.label_frac:.2f}.csv", index=False)
print(json.dumps({k: result[k] for k in result if k != "history"}, indent=2))


## 체크리스트
1. NCT + CRC-VAL + `nct_split_seed42.json` (+ SSL 실험 조건이면 Acc-1 `final.pt`) Dataset Add Input  
2. `CFG.run_tag` / `CFG.seed` / `CFG.label_frac`  
3. 로그: `copied/loaded split` (wrote만이면 Acc-1 split 미입력), SSL 실험 조건은 `missing_backbone 0`  
4. `monitor=holdout_val`, `test=crc_val` 확인  
5. `outputs/metrics/ft_*.json` 생성 후 Acc-3 집계
